# Tool과 Sub-Agent
* **LLM 토론 그래프**에 **Tool**과 **Sub-Agent**를 연결합니다.

---
* `day15` Conda 환경을 사용합니다.
* 웹 검색·주가 조회 패키지를 추가로 설치합니다.

In [ ]:
!pip install langchain-openai ddgs yfinance

In [ ]:
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()

print('WORKDIR :', WORKDIR)

---
## Tool 준비

| Tool | 출처 | 용도 |
|------|------|------|
| `web_search` | DuckDuckGo | 최신 뉴스·근거 검색 |
| `get_yf_stock_history` | yfinance | 티커별 최근 주가 |

In [ ]:
import yfinance as yf
from ddgs import DDGS
from ddgs.exceptions import DDGSException
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """최신 뉴스·시사·웹 정보를 검색합니다. 질문이나 키워드를 넣으세요."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
    except DDGSException:
        # ddgs는 결과 0건·일시 차단 시 예외를 던짐 → LLM에게 문자열로 전달
        return f'검색 결과 없음 (query: {query})'
    if not results:
        return '검색 결과 없음'
    return json.dumps(results, ensure_ascii=False)


@tool
def get_yf_stock_history(ticker: str, period: str = '1mo') -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


print(web_search.name)
print(get_yf_stock_history.name)

In [ ]:
# Tool 단독 테스트
print('=== 웹 검색 ===')
print(web_search.invoke('AI chip market 2026')[:400])
print()
print('=== 주가 조회 ===')
print(get_yf_stock_history.invoke({'ticker': 'NVDA', 'period': '5d'}))

---
## 웹 검색 근거 토론

15.3의 `optimist` / `skeptic` 토론에 **DuckDuckGo**를 붙입니다.

Tool을 쓰는 방법은 크게 두 가지입니다.

| 방식 | 핵심 | 장점 |
|------|------|------|
| **A. `bind_tools`** | Node 안에서 `tool_calls` 확인 후 직접 실행 | 코드가 한곳에 모임, 토론 그래프에 붙이기 쉬움 |
| **B. `ToolNode`** | Tool 실행을 **별도 Node**로 분리 | ReAct 패턴이 그래프에 그대로 드러남 |

* Tool 호출 시 State 필드명은 `chat_history` 대신 LangChain 관례인 **`messages`** 를 씁니다. (`add_messages`는 동일)

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage # -> Tool 사용과 관련된 메시지
from langchain_openai import ChatOpenAI


class ToolDebateState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 4
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


debate_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=OPENAI_API_KEY)
TOOLS = [web_search]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

### 방식 A — `bind_tools` + Node 내부 Tool 루프

1. `llm.bind_tools(tools)` 로 모델에 Tool 목록 전달
2. `AIMessage.tool_calls` 가 있으면 Tool 실행 → `ToolMessage` 추가 → LLM 재호출
3. `tool_calls` 가 없을 때 최종 발언으로 토론 history에 반영

In [ ]:
def run_llm_with_tools(llm_with_tools, prompts: list, max_rounds: int = 3):
    """Node 안에서 Tool 호출 루프를 돌리는 헬퍼"""
    msgs = list(prompts) # 입력받은 프롬프트를 리스트화해서

    for _ in range(max_rounds): # 최대 round까지 반복
        ai_msg = llm_with_tools.invoke(msgs) # llm에 지금까지 대화를 넣어 응답을 받고
        msgs.append(ai_msg)
        if not ai_msg.tool_calls: # tool 사용 요청이 없으면 바로 응답 반환
            return ai_msg 
        for call in ai_msg.tool_calls: # tool 사용 요청이 있을 경우, 도구를 실행하고 ToolMessage로 history에 추가
            tool_fn = TOOLS_BY_NAME[call['name']]
            result = tool_fn.invoke(call['args'])
            msgs.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    return ai_msg


optimist_llm_tools = debate_llm.bind_tools(TOOLS) # 기본 llm에 tool 정보를 binding (llm_with_tools에 들어갈 객체)
skeptic_llm_tools = debate_llm.bind_tools(TOOLS)

In [ ]:
def optimist_bind_node(state: ToolDebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 낙관적인 토론자입니다. 낙관적인 입장으로 토론에 참여하세요.'
            '상대방이 의견을 낸 경우, 적극적으로 반박하세요. 필요한 경우, 웹 검색 도구(web_search)를 사용할 수 있습니다.'
            '검색 결과를 인용해 3문장 이내로 답하세요.'
        )),
    ]
    if not state.messages:
        prompts.append(HumanMessage(content=f'토론 주제: {state.topic}'))
    else:
        prompts.extend(state.messages)

    response = run_llm_with_tools(optimist_llm_tools, prompts)
    response.name = 'optimist'
    return {
        'messages': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    }


def skeptic_bind_node(state: ToolDebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 비관적인 토론자입니다.'
            '비관적인 입장으로 토론에 참여하세요. 필요한 경우, 웹 검색 도구(web_search)를 사용할 수 있습니다.'
            '3문장 이내. 패배를 인정한다면 "합의"라고 말하세요.'
        )),
        *state.messages,
    ]
    response = run_llm_with_tools(skeptic_llm_tools, prompts)
    response.name = 'skeptic'
    return {
        'messages': [response],
        'last_speaker': 'skeptic',
    }

In [ ]:
from langgraph.graph import StateGraph, START, END


def tool_debate_route(state: ToolDebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.messages[-1].content if state.messages else ''
    if '합의' in last_text:
        return END
    return 'optimist_bind'


bind_workflow = StateGraph(ToolDebateState)
bind_workflow.add_node('optimist_bind', optimist_bind_node)
bind_workflow.add_node('skeptic_bind', skeptic_bind_node)
bind_workflow.add_edge(START, 'optimist_bind')
bind_workflow.add_edge('optimist_bind', 'skeptic_bind')
bind_workflow.add_conditional_edges('skeptic_bind', tool_debate_route)

bind_debate_app = bind_workflow.compile()

In [ ]:
for chunk in bind_debate_app.stream(ToolDebateState(topic='K팝 산업은 더욱 성장할 것이다')):
    print(chunk)

### 방식 B — `ToolNode` + `tools_condition`

* **agent Node**: `bind_tools` LLM이 `tool_calls` 를 내놓을 수 있음
* **tools Node**: `ToolNode`가 실제 검색 실행
* **tools_condition**: `tool_calls` 있으면 `tools` Node, 없으면 `END`

In [ ]:
from langgraph.prebuilt import ToolNode


class ReactState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'
    

In [ ]:
react_llm = debate_llm.bind_tools([web_search])


def react_optimist_node(state: ReactState) -> dict:
    prompts = [SystemMessage(content=(
        '당신은 낙관적 토론자입니다. 낙관적 입장에서 토론에 참여하세요.'
        'web_search tool을 활용해 근거를 검색할 수 있습니다.'
    )), ]

    if not state.messages:
        prompts.append(HumanMessage(content=f'토론 주제: {state.topic}'))
    else:
        prompts.extend(state.messages)
    
    response = react_llm.invoke(prompts)
    return {'messages': [response], 'last_speaker': 'optimist'}

def react_skeptic_node(state: ReactState) -> dict:
    prompts = [SystemMessage(content=(
        '당신은 비관적 토론자입니다. 비관적 입장에서 토론에 참여하세요.'
        'web_search tool을 활용해 근거를 검색할 수 있습니다.'
    )), ]

    prompts.extend(state.messages)

    response = react_llm.invoke(prompts)
    return {'messages': [response], 'last_speaker': 'skeptic', 'turn_count': state.turn_count + 1}


In [ ]:
react_workflow = StateGraph(ReactState)

react_workflow.add_node('optimist', react_optimist_node)
react_workflow.add_node('skeptic', react_skeptic_node)
react_workflow.add_node('tools', ToolNode([web_search]))


react_workflow.add_edge(START, 'optimist')

def route_after_optimist(state: ReactState):
    if state.messages[-1].tool_calls:
        return 'tools'
    return 'skeptic'

def route_after_skeptic(state: ReactState):
    if state.messages[-1].tool_calls:
        return 'tools'
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.messages[-1].content or ''
    if '합의' in last_text:
        return END
    return 'optimist'

def route_after_tools(state: ReactState):
    if state.last_speaker == 'optimist':
        return 'optimist'
    else:
        return 'skeptic'

react_workflow.add_conditional_edges('optimist', route_after_optimist)
react_workflow.add_conditional_edges('skeptic', route_after_skeptic)
react_workflow.add_conditional_edges('tools', route_after_tools)

react_app = react_workflow.compile()

In [ ]:
init_state = ReactState(
    topic="2024-2025 전기차 판매 통계를 바탕으로 전기차의 미래에 대해 토론"
)
for chunk in react_app.stream(init_state):
    print(chunk)

---
## 주식 도우미 Agent — 매수 vs 매도 토론

* `trading_data/mock_portfolio.json` 의 보유 종목을 참고합니다.
* **Bull(긍정)**: 매수·보유 정당화 | **Bear(부정)**: 매도·관망 정당화
* Tool: `web_search` + `get_yf_stock_history`

In [ ]:
class StockDebateState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str
    ticker: str = 'NVDA'
    turn_count: int = 0
    max_turns: int = 2


STOCK_TOOLS = [web_search, get_yf_stock_history]
STOCK_TOOLS_BY_NAME = {t.name: t for t in STOCK_TOOLS}
stock_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.5, api_key=OPENAI_API_KEY)

In [ ]:
def run_stock_llm_with_tools(llm_with_tools, prompts: list, max_rounds: int = 4):
    msgs = list(prompts)
    for _ in range(max_rounds):
        ai_msg = llm_with_tools.invoke(msgs)
        msgs.append(ai_msg)
        if not ai_msg.tool_calls:
            return ai_msg
        for call in ai_msg.tool_calls:
            tool_fn = STOCK_TOOLS_BY_NAME[call['name']]
            result = tool_fn.invoke(call['args'])
            msgs.append(ToolMessage(content=str(result)[:2000], tool_call_id=call['id']))
    return ai_msg


bull_llm = stock_llm.bind_tools(STOCK_TOOLS)
bear_llm = stock_llm.bind_tools(STOCK_TOOLS)

In [ ]:
def bull_node(state: StockDebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            f'당신은 매수·보유 찬성(Bull) 애널리스트입니다. 분석 종목: {state.ticker}.\n'
            'web_search와 get_yf_stock_history로 근거를 찾아 매수/보유를 정당화하세요. 3문장 이내.'
            '매도/관망 입장에 대해서는 적극적으로 반박하세요.'
            '참고: 지금은 2026년 7월입니다.'
        )),
        HumanMessage(content=state.question),
        *state.messages,
    ]
    response = run_stock_llm_with_tools(bull_llm, prompts)
    response.name = 'bull'
    return {'messages': [response], 'turn_count': state.turn_count + 1}


def bear_node(state: StockDebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            f'당신은 매도·관망 찬성(Bear) 애널리스트입니다. 분석 종목: {state.ticker}.\n'
            'Tool로 리스크 근거를 찾아 매도/관망을 정당화하세요. 3문장 이내. '
            '매수/보유 입장에 대해서는 적극적으로 반박하세요.'
            '참고: 지금은 2026년 7월입니다.'
            '양보 시 "합의"라고 말하세요.'
        )),
        HumanMessage(content=state.question),
        *state.messages,
    ]
    response = run_stock_llm_with_tools(bear_llm, prompts)
    response.name = 'bear'
    return {'messages': [response]}

In [ ]:
def stock_debate_route(state: StockDebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.messages[-1].content if state.messages else ''
    if '합의' in last_text:
        return END
    return 'bull'


stock_workflow = StateGraph(StockDebateState)
stock_workflow.add_node('bull', bull_node)
stock_workflow.add_node('bear', bear_node)
stock_workflow.add_edge(START, 'bull')
stock_workflow.add_edge('bull', 'bear')
stock_workflow.add_conditional_edges('bear', stock_debate_route)

stock_debate_app = stock_workflow.compile()

In [ ]:
user_question = 'NVDA를 더 매수해야 할까, 일부 매도해야 할까?'

init_state = StockDebateState(
    question=user_question,
    ticker='NVDA',
)

for chunk in stock_debate_app.stream(init_state):
    print(chunk)

---
## Sub-Agent — Main이 Sub 토론을 호출

* **Main Agent**: 질문 난이도 분류 → 단순 질문은 바로 답변
* **Sub-Agent**: 투자 판단처럼 **깊은 사고**가 필요하면 Step 2 토론 그래프를 **내부 호출**
* **Synthesize**: Sub 토론 결과를 바탕으로 최종 결론

In [ ]:
class MainAgentState(BaseModel):
    question: str
    needs_debate: bool = False
    route_reason: str = ''
    debate_transcript: str = ''
    final_answer: str = ''

In [ ]:
DEEP_KEYWORDS = ['매수', '매도', '팔', '사', '보유', '전망', '어떻게 생각', '해야 할까']


def triage_node(state: MainAgentState) -> dict:
    """간단한 규칙으로 Sub-Agent 필요 여부 판단"""
    q = state.question
    needs = any(k in q for k in DEEP_KEYWORDS)
    reason = '투자 판단 키워드 감지 → Sub 토론 필요' if needs else '단순 정보 질문 → 바로 답변'
    return {'needs_debate': needs, 'route_reason': reason}


def simple_answer_node(state: MainAgentState) -> dict:
    """단순 질문: 주가 Tool만 써서 빠르게 답변"""
    llm_tools = stock_llm.bind_tools([get_yf_stock_history])
    response = run_stock_llm_with_tools(llm_tools, [
        SystemMessage(content='한국어로 2문장 이내 답하세요.'),
        HumanMessage(content=state.question),
    ])
    return {'final_answer': response.content}


def debate_subagent_node(state: MainAgentState) -> dict:
    """Sub-Agent: 토론 그래프를 그대로 invoke"""
    sub_result = stock_debate_app.invoke(StockDebateState(question=state.question, ticker='NVDA'))
    lines = [f"[{m.name}] {m.content}" for m in sub_result['messages']]
    transcript = '\n'.join(lines)
    return {'debate_transcript': transcript}


def synthesize_node(state: MainAgentState) -> dict:
    """Sub 토론 기록을 읽고 최종 투자 의견 정리"""
    response = stock_llm.invoke([
        SystemMessage(content=(
            '당신은 수석 애널리스트입니다. Bull/Bear 토론을 읽고 '
            '매수/매도/관망 중 하나와 이유를 4문장 이내 한국어로 정리하세요.'
        )),
        HumanMessage(content=f"질문: {state.question}\n\n토론:\n{state.debate_transcript}"),
    ])
    return {'final_answer': response.content}

In [ ]:
def main_route(state: MainAgentState):
    return 'debate_subagent' if state.needs_debate else 'simple_answer'


def after_simple_or_debate(state: MainAgentState):
    return 'synthesize' if state.needs_debate else END


main_workflow = StateGraph(MainAgentState)
main_workflow.add_node('triage', triage_node)
main_workflow.add_node('simple_answer', simple_answer_node)
main_workflow.add_node('debate_subagent', debate_subagent_node)
main_workflow.add_node('synthesize', synthesize_node)

main_workflow.add_edge(START, 'triage')
main_workflow.add_conditional_edges('triage', main_route)
main_workflow.add_conditional_edges('simple_answer', after_simple_or_debate)
main_workflow.add_edge('debate_subagent', 'synthesize')
main_workflow.add_edge('synthesize', END)

main_app = main_workflow.compile()

### 단순 질문 — Sub-Agent 없이 바로 답변

In [ ]:
simple = main_app.invoke(MainAgentState(question='NVDA 최근 주가 추이 알려줘'))
print('route:', simple['route_reason'])
print('답변:', simple['final_answer'])

### 깊은 질문 — Sub-Agent 토론 후 결론

In [ ]:
deep = main_app.invoke(MainAgentState(question='NVDA를 지금 더 매수해야 할까?'))

print('route:', deep['route_reason'])
print('\n=== Sub 토론 기록 ===')
print(deep['debate_transcript'][:1200], '...' if len(deep['debate_transcript']) > 1200 else '')
print('\n=== 최종 결론 ===')
print(deep['final_answer'])

In [ ]:
for chunk in main_app.stream(MainAgentState(question='TSLA는 보유를 유지해야 할까 매도해야 할까?')):
    print(chunk)

## 실습

1. 아래 셀에서 `with_structured_output`으로 **`triage_node_llm`** 을 구현해 보세요. (기존 `triage_node`와 같은 `dict`를 반환)
   - 완성 후 `main_workflow`의 `triage` 노드를 `triage_node_llm`으로 바꿔 비교해 보세요.
2. Step 2 Bear/Bull 토론에 **moderator Node**를 추가해 매 라운드 요약 후 결론 내기
3. `input()` 루프로 사용자 질문을 받아 `main_app`을 반복 호출하는 채팅 UI 만들기

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


class TriageResult(BaseModel):
    needs_debate: bool = Field(description='심층 토론 필요 시 True')
    route_reason: str = Field(description='라우팅 이유 (한국어)')


triage_classifier = ChatOpenAI(
    model='gpt-4o-mini', temperature=0, api_key=OPENAI_API_KEY
).with_structured_output(TriageResult)


def triage_node_llm(state: MainAgentState) -> dict:
    # TODO: triage_classifier.invoke([SystemMessage(...), HumanMessage(...)])
    # TODO: return result.model_dump()
    pass